![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 02: Prompt Engineering and Retrieval-Augmented Generation)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- Teaching content is licensed under CC BY 4.0 and code under MIT; see [LICENSING.md](../../LICENSING.md) for scope and exclusions.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-AI-lab](https://github.com/tulip-lab/agentic-AI-lab/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 2B: Prompt Engineering as a Control Loop

<div align="center">

<table>
<thead>
<tr>
<th><strong>Item</strong></th>
<th><strong>Description</strong></th>
</tr>
</thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Environment</td><td>Google Colab or local Jupyter</td></tr>
<tr><td align="left">Main output</td><td>A contract-checked JSON extraction prompt improved through a logged, one-change-at-a-time iteration cycle, with every failure diagnosed before it is fixed</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m02b-overview)
2. [Setup and Background](#m02b-setup)
3. [Core Concepts](#m02b-core-concepts)
4. [Guided Implementation](#m02b-guided-implementation)
5. [Testing and Analysis](#m02b-testing)
6. [Student Tasks](#m02b-student-tasks)
7. [Submission and Reflection](#m02b-submission)

---

<a id="m02b-overview"></a>

### 1. Overview and Learning Goals

In [M02A-Prompt-Foundations](M02A-Prompt-Foundations.ipynb) you learned to write a prompt as a specification with named fields and a measurable output contract, and you compared prompt variants one field at a time. This session turns that habit into a complete engineering method. Real prompts are almost never right on the first attempt; what separates engineering from guesswork is not getting version one right, it is what you do when version one is wrong. The method of this session is a *control loop*: define an observable target, draft a prompt, run it, inspect the output against the contract, diagnose the earliest failure, change exactly one thing, and run again - keeping a written log of every version so that your final prompt comes with evidence of why each part of it exists.

A thermostat is a helpful analogy. A thermostat does not try to be clever; it has a target temperature, a thermometer, and one actuator, and it repeatedly measures, compares and adjusts. Your target temperature is the output contract, your thermometer is a checker function that code can run, and your actuator is the prompt text. Two disciplines make the loop work, and both come from the thermostat. First, the target must be *observable*: "make the output nicer" gives the thermometer nothing to measure, while "the output parses as JSON with exactly these three keys" does. Second, adjust one knob at a time: if you change the wording, the examples and the temperature all at once and the output improves, you have learned nothing about which change mattered, and you cannot safely remove any of them later.

The second theme of this session is *diagnosis before revision*. When an output fails its check, the failure has a cause, and different causes need different remedies. This lab uses a three-way taxonomy. A *specification failure* means the prompt never stated the requirement precisely enough, and the remedy is to revise the prompt. An *evidence-use failure* means the needed information and the requirement were both present but the model misused them, and the remedy is to strengthen grounding, most often with worked examples. An *evidence-access failure* means the needed information was never in the context at all, and no rewording can fix it; the remedy is to supply the evidence, which is the job of retrieval and the subject of the next session. You will meet all three in the guided work and learn to name them from the failure trace alone.

By the end of this lab, you should be able to turn a vague request into an observable target with a machine-checkable output contract; write a checker function and treat it as the single source of truth about pass or fail; separate *observation* (what one prompt version actually produced) from *evaluation* (a controlled comparison between versions on identical cases); run and log an iteration cycle in which every revision changes one variable and answers one diagnosis; explain when a few-shot demonstration fixes a failure that additional instructions cannot; and classify failures as specification, evidence-use or evidence-access problems. The evidence-access category is the bridge to [M02C-Retrieval-Augmented-Generation](M02C-Retrieval-Augmented-Generation.ipynb), and the whole loop reappears inside the visual workflows of [M03B-Flowise-Chatbot-Prompt-Memory](../../M03-Context-Orchestration/Flowise/M03B-Flowise-Chatbot-Prompt-Memory.md).

<a id="m02b-setup"></a>

### 2. Setup and Background

The setup follows `M02A` exactly, so it should already feel familiar. Model access goes through the Google Gemini API, the key is loaded from an environment variable or, when live mode is explicitly enabled, a hidden `getpass` prompt and is never written into the notebook, and every model call passes through one helper function, `generate_text`, that works in two modes. In **live mode** your prompts go to the real model; in **offline mode** they are answered from responses recorded from a real instruction-tuned model when this lab was authored, so the whole notebook runs without a key and behaves deterministically. Live outputs may differ in wording from the recorded ones; the loop you practise is the same either way, and the recorded outputs were chosen because they show the most common real failure at each iteration step.

One small upgrade from `M02A`: the recorded responses are now keyed by *tuples* of markers rather than single markers. This lab runs the same announcement through several prompt versions, so a recording must recognise both which announcement is in the prompt and which version of the prompt is asking. An entry matches only when every marker in its tuple appears in the prompt, and the first matching entry wins, so more specific entries are listed first. It is still just a tape recording, and for prompts you write yourself in the student tasks it returns a clearly labelled placeholder rather than pretending to answer.

If you want live mode, create a free key at [Google AI Studio](https://aistudio.google.com), store it as a Colab Secret named `GOOGLE_API_KEY` with notebook access enabled, or set `ENABLE_LIVE_MODEL = True` and paste it into the hidden prompt. Never place a key in a code cell: notebooks get shared and committed, and a leaked key must be revoked.

In [ ]:
# The lab is offline-first so a clean Run all never waits for a network install
# or a secret prompt. Set this switch to True only when you want to compare the
# recorded baseline with a live Gemini response, then re-run from this cell.
ENABLE_LIVE_MODEL = False

if ENABLE_LIVE_MODEL:
    get_ipython().run_line_magic("pip", "install -q -U google-genai")
else:
    print("Offline mode selected; live SDK installation skipped.")


In [ ]:
import os
import json
import re
from getpass import getpass
from typing import Any, Dict, List, Optional


def load_secret_from_environment(env_name: str, ask_if_missing: bool = False) -> Optional[str]:
    """Load a secret from an environment variable without printing it.

    Same safe-configuration pattern as M01A and M02A: the secret stays outside
    the notebook source and is never echoed. If the variable is missing and
    ask_if_missing is True, the user is prompted once through getpass;
    pressing Enter keeps the notebook in offline mode.
    """
    value = os.environ.get(env_name)
    if value:
        return value
    if ask_if_missing:
        typed = getpass(f"Enter {env_name} (press Enter to work offline): ")
        if typed:
            os.environ[env_name] = typed
            return typed
    return None


GOOGLE_API_KEY = load_secret_from_environment(
    "GOOGLE_API_KEY", ask_if_missing=ENABLE_LIVE_MODEL
)

# LIVE_MODE controls every model call in this notebook.
# True  -> prompts are sent to the Gemini API.
# False -> prompts are answered from recorded example outputs (offline mode).
LIVE_MODE = ENABLE_LIVE_MODEL and GOOGLE_API_KEY is not None

print("Live model mode:", LIVE_MODE)

The next cell defines the recorded responses. Read the entries top to bottom and you will see the whole story of this lab in miniature: the same extraction task answered by five prompt versions, each recording showing the characteristic failure (or success) that the corresponding version produced when this lab was authored. The entries are ordered from most specific to least specific, because the first entry whose markers *all* appear in the prompt wins.

In [ ]:
# Recorded example outputs for offline mode.
# Each entry is (markers, recorded_response). "markers" is a tuple of phrases
# that must ALL appear in the prompt for the entry to match. Entries are
# ordered most-specific first, because prompt version 5 also contains the
# wording of versions 2 to 4, and the first match wins.
MOCK_RESPONSES: List[tuple] = [
    # Version 5 (few-shot examples) on each of the three announcements.
    (("Examples:", "quiz on prompt engineering"),
     '{"module": "M02", "event": "quiz", "date": "2026-09-21"}'),
    (("Examples:", "exact date will be confirmed"),
     '{"module": "M03", "event": "workshop", "date": null}'),
    (("Examples:", "12 October 2026"),
     '{"module": "M05", "event": "assignment", "date": "2026-10-12"}'),

    # Version 4 (null rule stated as an instruction) on the no-date case:
    # the model wrote the string "null" instead of the JSON value null.
    (("value null", "exact date will be confirmed"),
     '{"module": "M03", "event": "workshop", "date": "null"}'),

    # Version 3 (raw-JSON contract) on the three announcements: the normal
    # case is clean, but the no-date and written-date cases drift.
    (("raw JSON", "quiz on prompt engineering"),
     '{"module": "M02", "event": "quiz", "date": "2026-09-21"}'),
    (("raw JSON", "exact date will be confirmed"),
     '{"module": "M03", "event": "workshop", "date": "unknown"}'),
    (("raw JSON", "12 October 2026"),
     '{"module": "M05", "event": "assignment", "date": "12 October 2026"}'),

    # Version 2 (basic JSON contract): valid JSON, but wrapped in a
    # markdown code fence - the most common real-world violation.
    (("Output contract", "quiz on prompt engineering"),
     '```json\n{"module": "M02", "event": "quiz", "date": "2026-09-21"}\n```'),

    # Version 1 (no contract at all): fluent, helpful, unusable prose.
    (("quiz on prompt engineering",),
     "The announcement is about the Module 02 quiz on prompt engineering. "
     "It opens on 21 September 2026 and covers the material from Sessions "
     "2A and 2B, so students should revise both prompt foundations and the "
     "control-loop method before attempting it."),
]

DEFAULT_MOCK_RESPONSE = (
    "[offline mode] No recorded response matches this prompt. Supply a "
    "GOOGLE_API_KEY for live answers, or add your own entry to MOCK_RESPONSES."
)


def mock_generate(prompt: str) -> str:
    """Return the first recorded response whose markers all appear in the prompt."""
    for markers, response in MOCK_RESPONSES:
        if all(marker in prompt for marker in markers):
            return response
    return DEFAULT_MOCK_RESPONSE


print("Recorded responses loaded:", len(MOCK_RESPONSES))

In [ ]:
# Model client and the single entry point for all model calls in this lab.
MODEL_NAME = "gemini-3.5-flash-lite"  # stable, fast and cost-conscious for lab comparisons.

model = None
if LIVE_MODE:
    from google import genai
    model = genai.Client(api_key=GOOGLE_API_KEY)


def generate_text(prompt: Any, temperature: float = 0.2) -> Dict[str, Any]:
    """Send a prompt to the model, or answer from recordings in offline mode.

    Returns the structured ok/error/result dictionary used throughout this
    unit, plus a "mode" field so every logged iteration records whether its
    evidence came from a live model or from a recording. The low default
    temperature matters more in this lab than in most: the control loop
    attributes output changes to prompt changes, so we hold the sampling
    knob still while we turn the prompt knob.
    """
    if not isinstance(prompt, str) or not prompt.strip():
        return {"ok": False, "error": "Prompt must be a non-empty string.",
                "result": None, "mode": "live" if LIVE_MODE else "mock"}

    if LIVE_MODE:
        try:
            response = model.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config={"temperature": temperature},
            )
            return {"ok": True, "error": None,
                    "result": response.text.strip(), "mode": "live"}
        except Exception as exc:  # network, quota or safety-block errors
            return {"ok": False, "error": f"API call failed: {exc}",
                    "result": None, "mode": "live"}

    return {"ok": True, "error": None,
            "result": mock_generate(prompt), "mode": "mock"}


smoke_test = generate_text("Reply with the single word: ready")
print("Mode:", smoke_test["mode"])
print("OK:", smoke_test["ok"])

In live mode you should see `Mode: live` and `OK: True`; authentication errors mean the key did not load, and quota errors mean the free tier needs a minute to recover. In offline mode you will see `Mode: mock`, and because the smoke-test prompt has no recorded markers its result would be the labelled placeholder - the honest behaviour we want from a recording. We also re-use the `build_prompt` specification builder from `M02A`, redefined below so this notebook stands alone.

In [ ]:
def build_prompt(task: str,
                 context: Optional[str] = None,
                 constraints: Optional[List[str]] = None,
                 examples: Optional[List[tuple]] = None,
                 output_contract: Optional[str] = None,
                 role: Optional[str] = None) -> str:
    """Assemble a prompt from explicit specification fields (from M02A).

    Every field except the task is optional, which is exactly what an
    iteration loop needs: each prompt version is a dictionary of fields, and
    moving from one version to the next means changing one entry. The
    labelled layout also means the prompt itself documents which fields it
    contains, so a logged version can be reconstructed and re-run later.
    """
    if not isinstance(task, str) or not task.strip():
        raise ValueError("A prompt specification requires a non-empty task.")

    parts: List[str] = []
    if role:
        parts.append(f"Role: {role}")
    parts.append(f"Task: {task}")
    if context:
        parts.append(f"Context:\n{context}")
    if constraints:
        bullet_list = "\n".join(f"- {c}" for c in constraints)
        parts.append(f"Constraints:\n{bullet_list}")
    if examples:
        demo_lines = "\n".join(f'"{text}" -> {label}' for text, label in examples)
        parts.append(f"Examples:\n{demo_lines}")
    if output_contract:
        parts.append(f"Output contract: {output_contract}")
    return "\n\n".join(parts)


print("build_prompt ready.")

<a id="m02b-core-concepts"></a>

### 3. Core Concepts

**The control loop.** A control loop is any system that repeatedly measures the gap between a target and reality, then acts to close it. Applied to prompting, the loop has five stations, and the arrows matter as much as the boxes: you may only travel from a failure back to a revision *through* a diagnosis.

```text
        +----------------------------------------------------------------+
        |                                                                |
        v                                                                |
+----------------+     +-----------+     +----------------------+     +--+----------+
| DRAFT / REVISE |     |    RUN    |     | INSPECT the output   |     |  DIAGNOSE   |
| prompt version | --> | the model | --> | against the output   | --> |  the        |
| (one change)   |     |           |     | contract (checker)   |     |  failure    |
+----------------+     +-----------+     +----------+-----------+     +-------------+
                                                    |
                                              contract met
                                                    |
                                                    v
                                       +------------------------+
                                       | log the result; run    |
                                       | the next test case or  |
                                       | stop and keep vN       |
                                       +------------------------+
```

The thermostat analogy pins down each part: the output contract is the target temperature, the checker function is the thermometer, the prompt text is the heater dial, and the log is the maintenance record. A thermostat that guessed the temperature by eye, or turned three dials at once, would never converge - and neither does a prompt engineer who eyeballs outputs and rewrites the whole prompt each round.

**Observable targets.** The loop starts before any prompt is written, with the question: *what would I measure to know this worked?* A vague request such as "summarise this announcement nicely" has no answer to that question. An observable target names the audience, the required content and a checkable limit - for example: "a JSON record that a program can read, with the module code, the event type from a fixed list, and the date in ISO format or null". Part of an observable target can be enforced by code; that part is the *output contract*, and in this lab it is strict JSON with exact keys and value formats, which a dozen lines of Python can check. Whatever cannot be checked by code (is the extracted event actually the right one?) is the *intent*, and Section 5 keeps the two judgements in separate columns, exactly as `M02A` did.

**Observation versus evaluation.** An *observation* is a recorded fact about one run: "version 3 produced `{'date': 'unknown'}` on the no-date announcement". An *evaluation* is a disciplined comparison: two prompt versions, identical test cases, identical checker, one variable changed - and a conclusion no broader than that comparison supports. The distinction protects you from the two classic self-deceptions of prompt work: generalising from one lucky run, and crediting the wrong change because several were made at once. The iteration log you build in Section 4 records observations; the analysis table in Section 5 performs evaluations.

**Diagnosis before revision.** When the checker reports a failure, resist the urge to immediately rewrite. First locate the cause, because the three causes have three different remedies, and applying the wrong remedy wastes an iteration (or worse, appears to work by luck).

```text
                        an output failed the check
                                    |
                 was the information needed for a correct
                 answer present in the prompt or context?
                        |                        |
                        no                       yes
                        |                        |
            EVIDENCE-ACCESS failure     did the prompt state the requirement
            the model cannot say what   precisely and checkably?
            it was never told                |                 |
            remedy: supply or retrieve      no                yes
            the evidence (M02C)              |                 |
                                 SPECIFICATION failure   EVIDENCE-USE failure
                                 the requirement was     requirement and evidence
                                 missing or ambiguous    were present but misused
                                 remedy: revise the      remedy: strengthen the
                                 prompt or contract      grounding - demonstrations,
                                                         delimiters, tighter format
```

The order of the questions matters: always rule out evidence-access first, because it is the one failure no prompt edit can repair. Asking a model for a quiz's assessment weighting when the announcement never mentions weighting will produce either a refusal or - far more dangerously - a fluent invention, and both have the same cause. Recognising that cause is what sends you to retrieval instead of another futile rewrite, and it is the single most important judgement this session trains.

**When examples beat instructions.** Specification failures are fixed by saying more; but there is a point where saying more stops working, and the guided work is engineered so you feel that point. Instructions *describe* a convention; demonstrations *show* it. Conventions about output shape - exact formatting, how to write a null, how to normalise a date - are cheap to show and surprisingly hard to describe, because the model must map your description onto tokens, and every mapping step can drift (you say "write null", it writes the string `"null"`). `M02A` showed that demonstrations define the task convention, for better or worse; this session gives the constructive corollary: when a failure survives a precise instruction, demonstrate the convention instead of describing it harder.

**The iteration log.** Every version you run gets a row: version number, the one change made, the cases run, and the result. The log is not bureaucracy; it is the difference between a prompt you can defend and a prompt that merely works today. The guided work below produces exactly this timeline:

```text
 v1 ---------> v2 ---------> v3 ---------> v4 ---------> v5
 task only     + output      contract      + null rule   + two worked
               contract      tightened:    stated as an  examples, one
               (JSON keys)   raw JSON,     instruction   showing a
                             no fences                   no-date case
 FAIL: fluent  FAIL: fenced  PASS normal   FAIL: writes  PASS normal,
 prose, not    JSON block    FAIL no-date  the string    no-date and
 JSON                        case          "null"        written-date
```

<a id="m02b-guided-implementation"></a>

### 4. Guided Implementation

The running task is small enough to hold in your head and realistic enough to matter: extract a structured record from a unit announcement, so that a downstream program (imagine a deadline-tracker agent from a later module) can consume it. The guided work follows the loop through five versions. For each version you will see the diagnosis of the previous failure, the single change made in response, the run, and the checker's verdict.

**Step 4.1: fix the target and build the thermometer.** We first pin down the observable target and write the checker, *before* writing any prompt. Working in this order is deliberate: the checker defines success, so writing it first stops you from unconsciously bending the definition of success towards whatever the model happens to produce. The target: given one announcement, produce a JSON object with exactly the keys `module` (a code such as `M02`), `event` (one of `quiz`, `workshop`, `assignment`, `lecture`) and `date` (ISO `YYYY-MM-DD`, or JSON `null` when the announcement states no full date). The three test announcements below cover a normal case, an edge case with no date, and a case with a date written in words that must be normalised.

In [ ]:
# The three fixed test cases. Holding the test set constant across every
# prompt version is what makes version-to-version comparison fair; if you
# change the cases and the prompt together, you can conclude nothing.
announcements: Dict[str, str] = {
    "normal": ("Reminder: the Module 02 quiz on prompt engineering opens on "
               "2026-09-21 and covers Sessions 2A and 2B."),
    "edge":   ("The Module 03 workshop on visual workflows will run in the "
               "usual lab; the exact date will be confirmed on the unit site."),
    "drift":  ("The Module 05 assignment briefing takes place on "
               "12 October 2026 during the Wednesday session."),
}

# What a correct extraction looks like for each case. This is the *intent*:
# the checker below can verify shape, but only this ground truth can verify
# meaning, and Section 5 reports the two judgements separately.
EXPECTED: Dict[str, Dict[str, Any]] = {
    "normal": {"module": "M02", "event": "quiz", "date": "2026-09-21"},
    "edge":   {"module": "M03", "event": "workshop", "date": None},
    "drift":  {"module": "M05", "event": "assignment", "date": "2026-10-12"},
}

for name, text in announcements.items():
    print(f"{name:7s} {text}")

In [ ]:
# The checker: the single source of truth about contract compliance.
ALLOWED_EVENTS = {"quiz", "workshop", "assignment", "lecture"}
MODULE_PATTERN = re.compile(r"^M0[1-8]$")
DATE_PATTERN = re.compile(r"^\d{4}-\d{2}-\d{2}$")


def check_record(model_output: Any) -> Dict[str, Any]:
    """Check a model response against the JSON extraction contract.

    Design decisions worth noticing:
    - json.loads does the parsing. It is safe (it only builds data, never
      runs code - which is why this unit bans unsafe execution) and it is the
      same call a downstream program would make, so the checker fails on
      exactly the outputs that would crash the consumer.
    - A markdown code fence is treated as a violation, not normalised away.
      M02A's check_label forgave capitalisation because no consumer cared;
      here the consumer is json.loads, and fences break it. The consumer
      defines what counts as harmless drift.
    - Every rejection names the violated rule, because the error text is the
      diagnostic evidence the control loop runs on.
    """
    if not isinstance(model_output, str) or not model_output.strip():
        return {"ok": False, "error": "Output is empty or not a string.",
                "result": None}

    cleaned = model_output.strip()
    if cleaned.startswith("```") or cleaned.endswith("```"):
        return {"ok": False,
                "error": "Output is wrapped in a markdown code fence.",
                "result": None}

    try:
        record = json.loads(cleaned)
    except json.JSONDecodeError as exc:
        return {"ok": False, "error": f"Output is not valid JSON: {exc}",
                "result": None}

    if not isinstance(record, dict):
        return {"ok": False, "error": "Top-level JSON value is not an object.",
                "result": None}

    expected_keys = {"module", "event", "date"}
    if set(record.keys()) != expected_keys:
        return {"ok": False,
                "error": f"Keys are {sorted(record.keys())}, "
                         f"expected {sorted(expected_keys)}.",
                "result": None}

    if not isinstance(record["module"], str) or not MODULE_PATTERN.match(record["module"]):
        return {"ok": False,
                "error": f"module must match M01..M08, got {record['module']!r}.",
                "result": None}

    if record["event"] not in ALLOWED_EVENTS:
        return {"ok": False,
                "error": f"event must be one of {sorted(ALLOWED_EVENTS)}, "
                         f"got {record['event']!r}.",
                "result": None}

    date = record["date"]
    if date is not None and (not isinstance(date, str) or not DATE_PATTERN.match(date)):
        return {"ok": False,
                "error": f"date must be YYYY-MM-DD or null, got {date!r}.",
                "result": None}

    return {"ok": True, "error": None, "result": record}


# A quick sanity run on hand-written strings, before any model is involved.
print(check_record('{"module": "M02", "event": "quiz", "date": "2026-09-21"}'))
print(check_record('```json\n{"module": "M02", "event": "quiz", "date": "2026-09-21"}\n```'))

The first hand-written string passes and comes back as a parsed dictionary ready for a program to use; the second fails with an error that names the fence. Notice what the checker gives the loop: not just a pass/fail bit, but a *reason*, and the reasons are worded to point at contract rules. When you diagnose failures in a moment, these error strings are your primary evidence.

**Step 4.2: build the flight recorder.** The iteration log is a list of rows, one per version run, and a helper that runs a version on named cases, checks every output, and appends the row. Bundling run-check-log into one function means no result can sneak past the record - the same honesty-by-construction argument behind `classify` in `M02A`.

In [ ]:
ITERATION_LOG: List[Dict[str, Any]] = []


def run_version(version: str, change: str, spec: Dict[str, Any],
                case_names: List[str], verbose: bool = True) -> Dict[str, Any]:
    """Run one prompt version on the named cases and log the outcome.

    "spec" is a dictionary of build_prompt fields (everything except the
    context, which carries the announcement). Passing specs as data rather
    than editing strings in place has a control-loop payoff: the difference
    between two versions is the difference between two dictionaries, so the
    "one change at a time" rule can be seen - and checked - at a glance.
    """
    outcomes: Dict[str, Any] = {}
    for name in case_names:
        prompt = build_prompt(context=f"Announcement: {announcements[name]}", **spec)
        response = generate_text(prompt)
        if response["ok"]:
            checked = check_record(response["result"])
        else:
            checked = {"ok": False, "error": response["error"], "result": None}
        outcomes[name] = {"raw": response["result"], "checked": checked,
                          "mode": response["mode"]}
        if verbose:
            print(f"[{version} | {name}] raw output:")
            print(f"    {response['result']!r}")
            verdict = "PASS" if checked["ok"] else f"FAIL - {checked['error']}"
            print(f"    checker: {verdict}")

    passed = sum(1 for o in outcomes.values() if o["checked"]["ok"])
    first_error = next((o["checked"]["error"] for o in outcomes.values()
                        if not o["checked"]["ok"]), None)
    ITERATION_LOG.append({
        "version": version,
        "change": change,
        "cases": ", ".join(case_names),
        "result": f"{passed}/{len(case_names)} passed"
                  + (f" - {first_error}" if first_error else ""),
    })
    return outcomes


def print_iteration_log() -> None:
    """Render the log as a fixed-width table: version, change made, result."""
    print(f"{'version':8s} {'change made':46s} {'cases':22s} {'result'}")
    print("-" * 118)
    for row in ITERATION_LOG:
        print(f"{row['version']:8s} {row['change'][:46]:46s} "
              f"{row['cases']:22s} {row['result']}")


print("Iteration log ready.")

**Step 4.3: version 1 - the naive draft.** Every loop needs a starting point, and starting simple is a feature: version 1 measures what the task alone buys you, so every later field must earn its place against this baseline. We run it on the normal case only - there is no point testing edge cases while the normal case still fails.

In [ ]:
SPEC_V1 = {
    "task": ("Extract the module code, the event type and the event date "
             "from the announcement."),
}

v1 = run_version("v1", "baseline: task and announcement only",
                 SPEC_V1, ["normal"])

The output is fluent, accurate English - and a total contract failure, because there is no contract. The checker reports that the output is not valid JSON, which under the taxonomy is a *specification failure*: nothing in the prompt ever asked for JSON, so the model defaulted to helpful prose. The model did nothing wrong; the specification was silent. Remedy: revise the prompt - add an output contract. That is one change, and it is the only change version 2 makes.

**Step 4.4: version 2 - add the output contract.**

In [ ]:
SPEC_V2 = dict(SPEC_V1)   # copy, never mutate - old versions must stay re-runnable
SPEC_V2["output_contract"] = (
    'Respond with a valid JSON object with exactly the keys "module", '
    '"event" and "date".'
)

v2 = run_version("v2", "added JSON output contract with exact keys",
                 SPEC_V2, ["normal"])

Progress and a new failure at once, which is the normal texture of the loop. The payload is now valid JSON with the right keys - the contract worked - but the model wrapped it in a ```` ```json ```` markdown fence, a habit instruction-tuned models learned because fenced code renders nicely in chat interfaces. The checker rejects it because `json.loads` would reject it. Diagnosis: still a *specification failure*, but a subtler one - the contract said "JSON" and the model produced JSON *presented for a human reader*. The specification never said the output would be read by a program. Remedy: tighten the contract wording to exclude the presentation layer. Again one change.

**Step 4.5: version 3 - tighten the contract, then probe the edges.** Version 3 rewrites the contract to demand raw JSON with no fences and no commentary, and pins down the value formats while we are there (one *field* changed - the contract - though it now carries the full format specification). Because the normal case is expected to pass now, this is also the moment to widen the test set: a version that survives its normal case has earned an edge-case probe, and the no-date announcement is the obvious stress test.

In [ ]:
SPEC_V3 = dict(SPEC_V2)
SPEC_V3["output_contract"] = (
    'Respond with a raw JSON object only - no markdown code fences, no '
    'commentary. Use exactly the keys "module", "event" and "date". '
    '"module" is a module code such as "M02". "event" is one of: quiz, '
    'workshop, assignment, lecture. "date" is the event date in '
    'YYYY-MM-DD format.'
)

v3 = run_version("v3", "contract tightened: raw JSON, no fences, formats",
                 SPEC_V3, ["normal", "edge"])

The normal case now passes end to end: raw JSON, correct keys, correct formats, and (check it against `EXPECTED["normal"]`) the right values. The edge case fails in an instructive way: asked for a date the announcement does not contain, the model filled the slot with `"unknown"`. Pause on the diagnosis, because this is where the decision tree earns its keep. Was the needed information present? Yes - the announcement genuinely states that no date is fixed, and recognising absence *is* the correct answer, so this is not evidence-access. Did the prompt state the requirement? No - the contract defines the format of a date that exists but says nothing about what to do when none does. *Specification failure*, third variety: an unhandled case rather than a missing or vague rule. Remedy: revise the prompt - state the null convention. Version 4 adds exactly that, as a constraint, together with the date-normalisation rule that the drift case will need later.

**Step 4.6: version 4 - state the null rule as an instruction.**

In [ ]:
SPEC_V4 = dict(SPEC_V3)
SPEC_V4["constraints"] = [
    ('If the announcement does not state a full calendar date, write the '
     'JSON value null for "date".'),
    "Convert any written date to YYYY-MM-DD format.",
]

v4 = run_version("v4", "added null rule and date normalisation as instructions",
                 SPEC_V4, ["edge"])

In the recorded run - and in a large fraction of live runs - version 4 fails the edge case in a way beginners find maddening: the model writes `"date": "null"`, the *string* "null", not the JSON value `null`. The instruction was understood well enough to change behaviour but not precisely enough to change it correctly. Run the decision tree once more. Information present? Yes. Requirement stated? Yes - version 4 states it explicitly and precisely. So this is no longer a specification failure: it is an *evidence-use failure*. The prompt described the convention; the model's mapping from description to tokens drifted at the last step, where "write null" is one character-level slip away from writing a quoted null.

This is the moment Section 3 promised: a failure that survives a precise instruction. You could try describing harder - "null without quotation marks, the JSON literal, not a string" - and it sometimes works, at the cost of an ever-longer contract that later models may read less carefully. The engineering remedy for an evidence-use failure is to strengthen grounding, and the cheapest strong grounding is a demonstration: *show* one announcement without a date mapped to a record with a bare `null`, and the convention stops being a description to interpret and becomes a pattern to continue. Version 5 adds two worked examples - one demonstrating date normalisation from written English, one demonstrating the null convention - and changes nothing else.

**Step 4.7: version 5 - demonstrate the convention.**

In [ ]:
DEMONSTRATIONS = [
    # One example demonstrates normalising a written date to ISO format...
    ("The Module 01 lecture on API safety moved to 3 August 2026.",
     '{"module": "M01", "event": "lecture", "date": "2026-08-03"}'),
    # ...and one demonstrates the null convention for a missing date.
    ("The Module 04 quiz will run at a time to be announced.",
     '{"module": "M04", "event": "quiz", "date": null}'),
]

SPEC_V5 = dict(SPEC_V4)
SPEC_V5["examples"] = DEMONSTRATIONS

v5 = run_version("v5", "added two worked examples (ISO date, null case)",
                 SPEC_V5, ["normal", "edge", "drift"])

All three cases pass, including the drift case, whose written date "12 October 2026" is normalised to `2026-10-12` - the first demonstration carried that convention, so the instruction from version 4 finally had a worked pattern to anchor it. Note what the examples did *not* replace: the contract and constraints from versions 3 and 4 are all still in the spec. Demonstrations and instructions are complements - the instruction states the rule, the example grounds it - and the log you are about to print shows exactly which failure each one answered.

**Step 4.8: read the flight record.**

In [ ]:
print_iteration_log()

Read the table bottom to top and every field of the final prompt has a documented reason to exist: the examples answer v4's evidence-use failure, the constraints answer v3's unhandled edge, the raw-JSON contract answers v2's fence, the contract itself answers v1's prose. That is what this session means by a prompt you can defend. It also tells you what you may safely *remove*: any field without a failure behind it is a candidate for deletion, and deleting untested cargo is how prompts stay short enough to maintain.

One diagnosis from the taxonomy has not appeared yet, because no prompt edit can make it appear: the *evidence-access failure*. Suppose the downstream deadline tracker also wants the quiz's weighting in the final grade. No announcement above mentions weighting. You could extend the schema and re-run all five versions; a live model will either leave the field null (if you are lucky and your null convention generalises) or - the dangerous outcome - invent a plausible percentage with perfect JSON shape, the exact silent-failure pattern of `M02A`'s flipped demonstrations. The information simply is not in the context, so the remedy is not iteration eight, nine and ten; it is to *supply the evidence* - retrieve the assessment policy document and place it in the context. Building that retrieval step properly is the whole of [M02C](M02C-Retrieval-Augmented-Generation.ipynb), and Section 5 gives you a failure trace of this kind to classify.

<a id="m02b-testing"></a>

### 5. Testing and Analysis

As in `M02A`, testing splits into two layers and we keep them apart. The *plumbing* - `check_record`, `build_prompt`, `run_version`'s logging, the validation in `generate_text` - is deterministic, so it gets hard `assert` statements: these are the mandatory tests and they pass with or without an API key. The *model behaviour* is probabilistic in live mode, so it gets a reported comparison table rather than asserts.

<div align="center">

<table>
<thead>
<tr>
<th><strong>Test type</strong></th>
<th><strong>Layer</strong></th>
<th><strong>Example in this notebook</strong></th>
</tr>
</thead>
<tbody>
<tr><td align="left">Normal case</td><td>Plumbing (assert)</td><td>A well-formed record string passes <code>check_record</code> and parses to a dictionary; a null date is accepted.</td></tr>
<tr><td align="left">Edge case</td><td>Plumbing (assert)</td><td>Surrounding whitespace is tolerated; key order does not matter.</td></tr>
<tr><td align="left">Failure case</td><td>Plumbing (assert)</td><td>Fenced JSON, prose, wrong keys, bad module codes, unknown events, malformed dates, empty and non-string input are all rejected with named reasons, never exceptions.</td></tr>
<tr><td align="left">Normal case</td><td>Behaviour (report)</td><td>v5 extracts the normal announcement correctly.</td></tr>
<tr><td align="left">Edge case</td><td>Behaviour (report)</td><td>v5 returns a JSON <code>null</code> for the no-date announcement; v3 does not.</td></tr>
<tr><td align="left">Failure case</td><td>Behaviour (report)</td><td>v3 on the written-date announcement copies the date verbatim - contract-violating format drift the table must surface.</td></tr>
</tbody>
</table>

</div>

In [ ]:
# ---- Plumbing tests: deterministic, so hard asserts are appropriate. ----
# These are the mandatory tests for this lab and they run identically in
# live and offline mode, because no model is involved.

# Normal case: a compliant record parses and passes.
good = check_record('{"module": "M02", "event": "quiz", "date": "2026-09-21"}')
assert good["ok"] is True and good["result"]["module"] == "M02"

# Normal case: the null-date convention is part of the contract.
null_date = check_record('{"module": "M03", "event": "workshop", "date": null}')
assert null_date["ok"] is True and null_date["result"]["date"] is None

# Edge case: harmless drift (whitespace, key order) is tolerated,
# because json.loads - the consumer - tolerates it.
assert check_record('  {"date": null, "event": "quiz", "module": "M02"}  ')["ok"] is True

# Failure case: a markdown fence is rejected with a named reason.
fenced = check_record('```json\n{"module": "M02", "event": "quiz", "date": null}\n```')
assert fenced["ok"] is False and "fence" in fenced["error"]

# Failure case: prose is rejected as invalid JSON.
assert check_record("The quiz opens on 21 September.")["ok"] is False

# Failure cases: structural violations are each caught and named.
assert check_record('{"module": "M02", "event": "quiz"}')["ok"] is False          # missing key
assert check_record('{"module": "M02", "event": "quiz", "date": null, "x": 1}')["ok"] is False  # extra key
assert check_record('{"module": "M99", "event": "quiz", "date": null}')["ok"] is False          # bad module
assert check_record('{"module": "M02", "event": "party", "date": null}')["ok"] is False         # unknown event
assert check_record('{"module": "M02", "event": "quiz", "date": "21/09/2026"}')["ok"] is False  # bad date format
assert check_record('{"module": "M02", "event": "quiz", "date": "null"}')["ok"] is False        # string "null"
assert check_record('[1, 2, 3]')["ok"] is False                                                 # not an object

# Failure case: hostile input is rejected safely, never raising.
assert check_record("")["ok"] is False
assert check_record(None)["ok"] is False
assert check_record(42)["ok"] is False

# Failure case: the plumbing around the model rejects bad prompts and specs.
assert generate_text("")["ok"] is False
try:
    build_prompt(task="   ")
    raise AssertionError("build_prompt should reject an empty task.")
except ValueError:
    pass

print("All plumbing tests passed.")

If the cell prints `All plumbing tests passed.`, the thermometer is trustworthy: every compliant record passes, every category of violation is caught with a named reason, and hostile input cannot crash the loop. A checker this thoroughly tested is what lets the behavioural table below make claims at all - if the checker were flaky, a "PASS" would mean nothing.

The behavioural comparison reruns versions 3 and 5 on all three cases and reports, per run, contract compliance *and* intent match (does the parsed record equal the ground truth?). This is an *evaluation* in the Section 3 sense: fixed cases, fixed checker, one difference between the versions (the demonstrations plus the null constraint they anchor). In offline mode the table reproduces the authored recordings exactly; in live mode expect v5 to pass everything nearly always, and v3 to fail the edge and drift cases in varying ways - variation which is itself an observation worth logging.

In [ ]:
# ---- Behavioural comparison: probabilistic in live mode, so report, not assert. ----

comparison_specs = {"v3": SPEC_V3, "v5": SPEC_V5}

print(f"{'version':8s} {'case':8s} {'contract':9s} {'intent':7s} {'raw output (truncated)'}")
print("-" * 100)
for version, spec in comparison_specs.items():
    for case in ["normal", "edge", "drift"]:
        outcome = run_version(f"{version}*", f"section 5 re-run of {version}",
                              spec, [case], verbose=False)[case]
        record = outcome["checked"]["result"]
        contract_ok = outcome["checked"]["ok"]
        intent_ok = (record == EXPECTED[case]) if contract_ok else False
        raw_short = str(outcome["raw"]).replace("\n", " ")[:44]
        print(f"{version:8s} {case:8s} {str(contract_ok):9s} {str(intent_ok):7s} {raw_short}")

print()
print("Note: 'contract' checks output shape; 'intent' checks the parsed record")
print("against the ground truth in EXPECTED. Rows marked v3*/v5* were appended")
print("to the iteration log too - re-runs are runs, and runs get logged.")

Two analysis habits to close the section. First, read the failures *by category, not by symptom*. The v3 edge failure (`"unknown"`) and the v3 drift failure (verbatim written date) look different in the raw column but share a diagnosis - unspecified conventions - and were both fixed by the same v5 change, which is why a diagnosis column is more valuable in a log than a symptom column. Second, practise the taxonomy on a trace you have not seen. Here are three failure traces from a hypothetical classmate's log for the same task; classify each as specification, evidence-use or evidence-access before reading on.

```text
 trace A: prompt asks for the event's assessment weighting as a fourth key;
          announcement never mentions weighting; model returns
          {"weighting": "20%"} with perfect JSON shape.
 trace B: prompt says 'date in YYYY-MM-DD format'; announcement contains
          '12 October 2026'; model returns {"date": "12 October 2026"}.
 trace C: prompt never mentions what "module" should look like; model
          returns {"module": "Module 02"} and the checker rejects it.
```

Trace A is the dangerous one: *evidence-access* - the weighting was never in the context, the confident `"20%"` is an invention, and the remedy is retrieval, not rewording. Trace B is *evidence-use* if your v4 experience generalises (the rule was stated; grounding was too weak) - a demonstration is the remedy. Trace C is a plain *specification failure*: the requirement was never stated, so state it. If you classified A as a specification failure and "fixed" it by adding "do not invent weightings", you would get a null - and still have a system that cannot answer the question, which is why the decision tree asks about evidence first.

<a id="m02b-student-tasks"></a>

### 6. Student Tasks

You now run the full loop on a task you design. Choose a small text-to-JSON extraction task with a closed schema of at least three keys, where at least one key is *optional in the source text* (so it needs a null convention, giving your loop a real edge case to converge on). Good examples: extracting `{"unit": ..., "room": ..., "time": ...}` from timetable change notices, or `{"title": ..., "severity": ..., "deadline": ...}` from maintenance notifications. Write your own three test inputs: one normal, one missing the optional field, one with a value that needs normalisation.

<div align="center">

<table>
<thead>
<tr>
<th><strong>Task</strong></th>
<th><strong>What you need to do</strong></th>
<th><strong>Why it matters</strong></th>
<th><strong>Expected evidence</strong></th>
</tr>
</thead>
<tbody>
<tr><td align="left">Task 1</td><td>Define your task as an observable target: audience (which program consumes the output), required content (the schema and value formats), and the checkable limit (your output contract). Write your three test inputs and their ground-truth records in an <code>EXPECTED</code>-style dictionary.</td><td>The loop cannot start without a measurable target; writing ground truth first stops you bending "correct" towards whatever the model produces.</td><td>A markdown cell stating the target, plus the test inputs and ground-truth dictionary in code.</td></tr>
<tr><td align="left">Task 2</td><td>Write <code>check_my_record(model_output)</code> following the <code>check_record</code> pattern, with named error messages per rule, and test it before any model call: normal (compliant record passes, null convention accepted), edge (whitespace and key order tolerated), and failure cases (fence, prose, wrong keys, bad value formats, empty string, non-string input) all returning <code>ok=False</code> without raising.</td><td>The checker is your thermometer; an untested thermometer makes every later "PASS" meaningless.</td><td>The checker function and passing assert output covering normal, edge and failure cases.</td></tr>
<tr><td align="left">Task 3</td><td>Run at least four logged iterations with <code>run_version</code> (or your own equivalent): each version changes exactly one spec field, each change answers a written diagnosis of the previous failure, and at least one iteration must fix a diagnosed failure with a demonstration where an instruction was already tried. Print the final iteration log.</td><td>This is the session's core skill: converging on a working prompt with evidence, not luck - and feeling where instructions stop working and examples take over.</td><td>The printed iteration log (version, change made, cases, result) plus a one-line diagnosis per failed version, in a markdown cell or comments.</td></tr>
<tr><td align="left">Task 4</td><td>Classify every failure from your Task 3 log as specification, evidence-use or evidence-access, and write a short analysis (100 to 200 words) justifying each classification and naming the matching remedy. Include at least one question your task <em>cannot</em> answer from its inputs, and state why its failure would be evidence-access.</td><td>Diagnosis is what makes the loop converge; the evidence-access case is the bridge to retrieval in M02C.</td><td>The written classification and analysis in a markdown cell.</td></tr>
</tbody>
</table>

</div>

For the programming work in Tasks 2 and 3, the expected behaviours are:

<div align="center">

<table>
<thead>
<tr>
<th><strong>Input case</strong></th>
<th><strong>Example</strong></th>
<th><strong>Expected behaviour</strong></th>
</tr>
</thead>
<tbody>
<tr><td align="left">Normal case</td><td>Your normal test input under your final prompt version</td><td>Contract-compliant JSON whose parsed record equals your ground truth.</td></tr>
<tr><td align="left">Edge case</td><td>Your input with the optional field missing</td><td>Contract-compliant JSON with the demonstrated null (or equivalent) convention.</td></tr>
<tr><td align="left">Failure case</td><td>An early prompt version on any input</td><td>Checker returns <code>ok=False</code> with a named reason; the run still appears in the log.</td></tr>
<tr><td align="left">Failure case</td><td><code>check_my_record("")</code> and <code>check_my_record(None)</code></td><td>Return <code>ok=False</code> with a clear error message, never raise an exception.</td></tr>
</tbody>
</table>

</div>

If you are working offline, `generate_text` will return the labelled placeholder for your new prompts. As in `M02A`, either add recorded entries to `MOCK_RESPONSES` for your own prompts (hand-write plausible outputs - including plausible *failures* for early versions - and mark them as hand-written in a comment), or reason through the loop on predicted outputs and state clearly that they are predictions. Hand-written failure recordings are not cheating; they are how you practise the diagnosis skill deterministically. Silently presenting placeholder text as model output is the only wrong option.

In [ ]:
# Student task starter.
# Task 1: define your observable target, test inputs and ground truth.

# TODO: describe your extraction task and its consumer in a markdown cell,
# then define your data here.
# my_inputs = {
#     "normal": "...",
#     "edge": "...",     # the optional field is missing here
#     "drift": "...",    # a value that needs normalisation
# }
# MY_EXPECTED = {
#     "normal": {...},
#     "edge": {...},
#     "drift": {...},
# }

# Task 2: write your contract checker following the check_record pattern.
# def check_my_record(model_output: Any) -> Dict[str, Any]:
#     ...

In [ ]:
# Student task tests.
# Uncomment and adapt after completing Tasks 1 and 2. The checker must be
# fully tested BEFORE you run your first prompt version.

# Normal case: a compliant record passes and parses.
# assert check_my_record('{...a valid record...}')["ok"] is True

# Normal case: your null/missing-field convention is accepted.
# assert check_my_record('{...record with null field...}')["ok"] is True

# Edge case: whitespace and key order are tolerated.
# assert check_my_record('  {...reordered keys...}  ')["ok"] is True

# Failure cases: each violation is rejected with a named reason, never raising.
# assert check_my_record('```json ... ```')["ok"] is False
# assert check_my_record("plain prose")["ok"] is False
# assert check_my_record('{...wrong keys...}')["ok"] is False
# assert check_my_record("")["ok"] is False
# assert check_my_record(None)["ok"] is False

# print("Student checker tests passed.")

# Task 3: run your logged iterations here. Reset the log first so your
# table contains only your own versions:
# ITERATION_LOG.clear()
# ... run_version("v1", "baseline", {...}, ["normal"]) etc. ...
# print_iteration_log()

<a id="m02b-submission"></a>

### 7. Submission and Reflection

Submit the completed notebook with the following evidence. Marking emphasis is on the discipline of the loop - one change per version, diagnosis before revision, honest logging - not on how quickly your prompt converged; a log with instructive failures is worth more than a log that passes on version two.

<div align="center">

<table>
<thead>
<tr>
<th><strong>Required item</strong></th>
<th><strong>What to submit</strong></th>
<th><strong>Quality check</strong></th>
</tr>
</thead>
<tbody>
<tr><td align="left">Observable target</td><td>Your Task 1 target statement, test inputs and ground-truth dictionary.</td><td>Names the consumer, the schema and the checkable limit; includes a normal, a missing-field and a normalisation input.</td></tr>
<tr><td align="left">Contract checker</td><td><code>check_my_record</code> and its tests.</td><td>Normal, edge and failure asserts all pass before any model call; every rejection names the violated rule; invalid input never raises.</td></tr>
<tr><td align="left">Iteration log</td><td>The printed Task 3 log.</td><td>At least four versions; exactly one spec change per version; each failed version followed by a one-line diagnosis; at least one failure fixed by a demonstration after an instruction was tried; states whether runs are live, recorded or predicted.</td></tr>
<tr><td align="left">Failure classification</td><td>The Task 4 analysis.</td><td>Every logged failure classified with a justification and matching remedy; includes one argued evidence-access case.</td></tr>
<tr><td align="left">Reflection</td><td>150 to 250 words.</td><td>Refers to concrete rows of your own log, not generic statements about prompting.</td></tr>
</tbody>
</table>

</div>

Reflection questions:

1. Which single change in your log produced the largest improvement, and what diagnosis did it answer?
2. Where in your loop did an instruction fail and a demonstration succeed - or, if that never happened, why might your task not have needed it?
3. Why must the checker be written and tested before the first prompt version is run?
4. What would break, methodologically, if you changed two spec fields between versions and the result improved?
5. Describe one question your extraction task cannot answer from its inputs. Why is more prompt iteration the wrong remedy, and what does that imply about the systems you will build in M02C?

Use the debugging guide below if the notebook does not behave as expected.

<div align="center">

<table>
<thead>
<tr>
<th><strong>Symptom</strong></th>
<th><strong>Likely cause</strong></th>
<th><strong>How to inspect</strong></th>
<th><strong>Typical fix</strong></th>
</tr>
</thead>
<tbody>
<tr><td align="left"><code>Live model mode: False</code> unexpectedly</td><td>Key not present in the environment</td><td>Re-run the key cell and check for typos in <code>GOOGLE_API_KEY</code></td><td>Set the Colab Secret or paste the key into the hidden prompt, then re-run the client cell</td></tr>
<tr><td align="left">Offline placeholder text in results</td><td>Your new prompt matches no recorded markers</td><td>Check whether the raw output starts with <code>[offline mode]</code></td><td>Add a <code>MOCK_RESPONSES</code> entry whose marker tuple identifies your prompt version and test case</td></tr>
<tr><td align="left">Wrong recording returned offline</td><td>A less specific marker tuple matched first</td><td>Print the prompt and check which entry's markers all appear in it</td><td>Reorder entries most-specific first, or add a distinguishing marker</td></tr>
<tr><td align="left">Live v5 still fails the edge case</td><td>Model overrode the demonstrations</td><td>Re-run a few times and count outcomes; inspect the raw output</td><td>Add a second no-date demonstration; if it persists, log it - instability is a finding</td></tr>
<tr><td align="left">Checker passes output that looks wrong</td><td>Contract narrower than your intent</td><td>Compare the parsed record against your ground truth by eye</td><td>That gap is intent, not contract: either extend the contract or report intent separately as in Section 5</td></tr>
<tr><td align="left"><code>API call failed</code> mentioning quota or 429</td><td>Free-tier rate limit reached</td><td>Read the error text in the returned dictionary</td><td>Wait a minute and re-run; run one version at a time</td></tr>
</tbody>
</table>

</div>

When you have completed the submission items, continue to [M02C: Retrieval-Augmented Generation](M02C-Retrieval-Augmented-Generation.ipynb), where the evidence-access failures your loop cannot fix become the problem statement, and retrieval becomes the remedy.

#### Further Readings

- Google Gemini structured output guide: <https://ai.google.dev/gemini-api/docs/structured-output>
- Google Gemini prompting strategies: <https://ai.google.dev/gemini-api/docs/prompting-strategies>
- OpenAI prompt engineering guide: <https://platform.openai.com/docs/guides/prompt-engineering>
- JSON Schema (formalising output contracts): <https://json-schema.org>
- Madaan et al., "Self-Refine: Iterative Refinement with Self-Feedback" (automating the loop): <https://arxiv.org/abs/2303.17651>
- White et al., "A Prompt Pattern Catalog to Enhance Prompt Engineering with ChatGPT" (reusable prompt structures): <https://arxiv.org/abs/2302.11382>
- Min et al., "Rethinking the Role of Demonstrations" (what demonstrations actually teach): <https://arxiv.org/abs/2202.12837>